### Steps
```
1. merge tables
2. keep cols only available in test(kaggle)
3. lowercase cols
4. drop when store closed
5. delete columns: open
6. school holiday: convert to int
7. state holiday: convert to str
8. extend date
```

# features
```
class FeatureEngineering:   # before split
class FeatureTransformer:   # after split (fit only on train)
class ModelTrainer
```

In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))
pd.set_option("display.max_columns", None)

from src.utils.basics_info import extend_date, distance_bucket

In [2]:
df_rossmann = pd.read_csv("../data/Rossmann Stores Data.csv")
df_store = pd.read_csv("../data/store.csv")
df_rossmann.merge(df_store, how="left", on="Store").head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [3]:
class FeatureEngineering:



    def merge_and_transform_dfs(self, df_rossmann:pd.DataFrame, df_store:pd.DataFrame, is_train:bool):
    
        print(f"shape rossmann: {df_rossmann.shape}")
        print(f"shape store: {df_store.shape}")
        df = df_rossmann.merge(df_store, how="left", on="Store")
        print(f"shape after merge: {df.shape}")
        cols_in_test = ['Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
        cols = cols_in_test.copy()
        if is_train:
            cols.append("Sales")
        df = df[cols] # Keep only those cols which are available in test only.
        print(f"shape with test cols only: {df.shape}")
        print("converting columns to lowercase...")
        df.columns = [col.lower().strip() for col in df.columns]
        return df
    
    
    def filter_open_stores(self, df:pd.DataFrame, cols2drop= None):

        if cols2drop is None:
            cols2drop = ["open"]
        print(f"shape: {df.shape}")
        if "open" in df.columns:
            df = df[df["open"]==1]
        print(f"shape: {df.shape}")
        df = df.drop(cols2drop, axis=1)
        print(f"shape: {df.shape}")
        return df
    
    
    def data_preprocessing(self, df:pd.DataFrame):
    
        print(f"shape: {df.shape}")
        df = extend_date(df, "date")
        if "year" in df.columns:
            df = df.drop("year", axis = 1)
        print(f"shape: {df.shape}")
        return df



class FeatureTransformer:


    def __init__(self):

        self.num_cols = ["month", "week_of_year"]
        self.scaler = None
        self.final_columns = None
        self.store_mapping = None


    def fit(self, df: pd.DataFrame):

        from sklearn.preprocessing import StandardScaler

        df = df.copy()

        # type conversions (safe here)
        df["stateholiday"] = df["stateholiday"].astype(str)
        df["schoolholiday"] = df["schoolholiday"].astype(int)

        # self.store_mapping = dict(enumerate(df["store"].astype("category").cat.categories))
        categories = df["store"].astype("category").cat.categories
        self.store_mapping = {v: k for k, v in enumerate(categories)}
        df["store"] = df["store"].map(self.store_mapping)
        # self.store_mapping = {v: k for k, v in self.store_mapping.items()}

        # df["store"] = df["store"].astype("category").cat.codes
        df = pd.get_dummies(df, columns=["stateholiday"], drop_first=True)

        self.scaler = StandardScaler()
        self.scaler.fit(df[self.num_cols])
        self.final_columns = df.columns

        return self


    def transform(self, df: pd.DataFrame):
        df = df.copy()

        df["stateholiday"] = df["stateholiday"].astype(str)
        df["schoolholiday"] = df["schoolholiday"].astype(int)
        df["store"] = df["store"].map(self.store_mapping).fillna(-1)
        df = pd.get_dummies(df, columns=["stateholiday"], drop_first=True)

        available_cols = [col for col in self.num_cols if col in df.columns]
        df[available_cols] = self.scaler.transform(df[available_cols])
        
        df = df.reindex(columns=self.final_columns, fill_value=0)

        return df


    

In [4]:
feature_eng = FeatureEngineering()
df = feature_eng.merge_and_transform_dfs(df_rossmann.copy(), df_store.copy(), is_train=True)
print("-"*100)
df = feature_eng.filter_open_stores(df)
print("-"*100)
df = feature_eng.data_preprocessing(df.copy())

shape rossmann: (1017209, 9)
shape store: (1115, 10)
shape after merge: (1017209, 18)
shape with test cols only: (1017209, 8)
converting columns to lowercase...
----------------------------------------------------------------------------------------------------
shape: (1017209, 8)
shape: (844392, 8)
shape: (844392, 7)
----------------------------------------------------------------------------------------------------
shape: (844392, 7)
shape: (844392, 10)


In [5]:
df.head()

,store,dayofweek,date,promo,stateholiday,schoolholiday,sales,month,day,week_of_year
0,1,5,2015-07-31,1,0,1,5263,7,31,31
1,2,5,2015-07-31,1,0,1,6064,7,31,31
2,3,5,2015-07-31,1,0,1,8314,7,31,31
3,4,5,2015-07-31,1,0,1,13995,7,31,31
4,5,5,2015-07-31,1,0,1,4822,7,31,31
